# AlexNet Paper 2014

## Objective
1. To a gain a better understanding of AlexNet image classification model introdued in the paper **[ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf)** and **[One weird trick for parallelizing convolutional neural networks](https://arxiv.org/pdf/1404.5997)**.

The original paper is "ImageNet Classification with Deep Convolutional Neural Networks" but pytorch implement's the version provided in the paper "**One weird trick for parallelizing convolutional neural networks**". The first paper had performed model training over 2 GPUs due to lack of VRAM on GPUs. The model provided in the second paper is a slightly different version than the original but it was trained over a single GPU

**Note**:
- To look at the entire model's implementation at one place you can have a look at the model's code written here`computer-vision/src/computer_vision/classification/alexnet.py`
- To understand image classification thoroughly look at the information provided under the LeNet 1 notebook `computer-vision/src/computer_vision/classification/lenet_1.ipynb`
- To understand dropout layesr in depth you can look at [Dropout Layer Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/dropout_layers.ipynb)
- To understand pooling layers in depth you can look at [Pooling Layers Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/pooling_layers.ipynb)
- To understand convolution neural network layer in depth you can look at [Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb) (**I advised you to go through this notebook before you go ahead for better understanding of convolution layers**)

## AlexNet Architecture and Training Pipeline
This is the first model that has been trained on a RGB coloured image dataset. From this model onwards you will see almost all of the models have been trained on RGB coloured images.

### Dataset
The dataset comprises of 1000 images for 1000 classes as part of the training dataset and 50,000 test images of shape (3, 224, 224)

### Model Architecture
Models of both papers have been implemented but we will look at the model provided in "**One weird trick for parallelizing convolutional neural networks**"

The AlexNet follows the same template as that followed and introduced in LeNet where we have convolution layers for feature extraction, pooling layers for sub sampling and fully connected layers for classification. This model introduces on top of it the **Drop Out Layer**. Dropout layers allow for better regularization compared to L2 and L1 regularization as they force every neuron to participate in learning features. These dropout layers are carried forward as part of the computer vision model template. The authors also introduce another component which they believe aids in learning better called the Local Response Normalization. This was proved to be not effective and hence dropped in modern neural network stack. This was replaced by the process of Batch Normalization. For now you need to not focus on it.

The model is divided into 2 blocks **feature** and **classifier** block.

#### Feature Block
The feature block comprises of 5 convolution layers, 3 pooling layers and 2 Local Response Norm Layers. **ReLU** activation function is used as the non linear activation function. The components will always be placed as follows:<br>
$$Conv -> ReLU -> LocalResponseNorm -> MaxPooling$$

##### 1. First Convoution Block
The input to this layer is of the shape $(N,3,224,224)$ it passes through a convolution layer with **3** sets of **64** convolution filters of kernel size **11** convolving with the stride of **4** and padding of **2**. Only the first layer of the 5 convolution layers reduces the size of the feature map. The final output shape becomes $(N,64,55,55)$. Thes feature maps go through the **ReLU** activation function layer into **Local Response Norm** layer. The final layer of the first convolution block part of the feature block is a **Max Pooling** layer which sub samples the feature map with filter of size **3**, stride **2** and padding **0**. We can see that the input goes from $(N,3,224,224)$ to $(N,64,27,27)$

##### 2. Second Convolution Block
This block has the same structure as that of the first block. The only difference is that in the second block the number of feature maps are increased by the convolution layer from **64** to **192**. The input goes from $(N,64,27,27)$ to $(N,192,13,13)$

##### 3. Third Convolution Block
This block only comparises of a convolution layer followed by a ReLU activation layer. The convolution layer increases the feature maps from **192** to **384** and has convolution filters of kernel size **3** convolving with a stride of **1** and padding of **1**. The input goes from $(N,192,13,13)$ to $(N,384,13,13)$

##### 4. Fourth Convolution Block
This block has the same structure as that of the third block. The only difference is that in the fourth block the number of feature maps are decreased by the convolution layer from **384** to **256**. The input goes from $(N,384,13,13)$ to $(N,256,13,13)$

**Note**: The paper says the number of feature maps remains the same **384** to **384** but in the configuration file that accompanies the paper it has mentioned to drop from **384** to **256**. Pytorch has followed the config file and so should we for simplification.

##### 5. Fifth Convolution Block
This block has a convolution layer followed by a ReLU layer followed by a max pooling layer. The max pooling layer has the same attributes as the ones described in first and second convolution block. The feature maps do not increase and remain at 256. The attributes of the convolution filter here is that they have a kernel size of **3** convolving with a stride of **1** and padding of **1**. The input goes from $(N,256,13,13)$ to $(N,256,6,6)$

#### Classifier Block
In this block there are are **3** fully connected layers.The components will always be placed as follows:<br>
$$Dropout -> Linear -> ReLU$$

 The first 2 layers follow **dropout** layers with a dropout probability of **50%** and every fully connected layer is followed by a **RELU** layer. The input to this block is flattened and the total input features become **9216**. The first fully connected layer takes these **9216** features and reduces them to **4096** features which pass through the second dropout layer into a fully connected layer which keeps the same amount of feature of **4096**. The last fully connect layer takes the 4096 feature and outputs to the desired number of classes in this case **1000**.

Below is the complete AlexNet's structure for 1000 classes

In [1]:
import torch
from torchinfo import summary
from computer_vision.classification.alexnet import AlexNetV2


summary(AlexNetV2(1000), input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
AlexNetV2                                [1, 3, 224, 224]          [1, 1000]                 --                        --
├─Sequential: 1-1                        [1, 3, 224, 224]          [1, 256, 6, 6]            --                        --
│    └─Conv2d: 2-1                       [1, 3, 224, 224]          [1, 64, 55, 55]           [11, 11]                  23,296
│    └─ReLU: 2-2                         [1, 64, 55, 55]           [1, 64, 55, 55]           --                        --
│    └─LocalResponseNorm: 2-3            [1, 64, 55, 55]           [1, 64, 55, 55]           --                        --
│    └─MaxPool2d: 2-4                    [1, 64, 55, 55]           [1, 64, 27, 27]           3                         --
│    └─Conv2d: 2-5                       [1, 64, 27, 27]           [1, 192, 27, 27]          [5, 5]                    307,392
│    └─ReL